# Mesure COSMIC (ISO/IEC 19761) — moteur kodoneko

**Auteur :** Benoît Moraillon · **Licence :** AGPL-3.0

---

## À quoi sert ce notebook

Ce notebook mesure la **taille fonctionnelle** d'un logiciel à partir de son
code source, selon la méthode **COSMIC** (norme ISO/IEC 19761). Il démontre le
moteur **kodoneko** sur des exemples simples puis sur de **vrais dépôts publics**
(Spring PetClinic, template FastAPI…).

> **En une phrase :** kodoneko lit le code, repère les échanges de données qui
> ont un sens fonctionnel, et en déduit une **estimation** de la taille COSMIC,
> accompagnée d'un **intervalle de confiance** et d'un détail auditable.

---

## 1. Qu'est-ce que la méthode COSMIC ?

COSMIC mesure la taille d'un logiciel en comptant ses **mouvements de données**
(*data movements*). L'idée : la taille fonctionnelle est proportionnelle au
nombre de fois où une donnée traverse une frontière du système.

Il existe **quatre types de mouvements**, et chacun vaut **1 CFP**
(*COSMIC Function Point*) :

| Type | Symbole | Définition | Exemple typique |
|------|:-------:|------------|-----------------|
| **Entry** | E | une donnée entre dans le système depuis un utilisateur | requête HTTP reçue, formulaire soumis |
| **Exit** | X | une donnée sort vers un utilisateur | réponse HTTP, page affichée, message |
| **Read** | R | une donnée est lue depuis un stockage persistant | `SELECT`, `findById`, lecture cache |
| **Write** | W | une donnée est écrite dans un stockage persistant | `INSERT`, `save`, écriture cache |

Le comptage se fait **par processus fonctionnel** (*functional process*) — en
pratique, le plus souvent **un endpoint = un processus fonctionnel**.

**Il n'y a aucune pondération de complexité** : un mouvement simple et un
mouvement compliqué valent tous les deux 1 CFP. C'est ce qui rend la méthode
reproductible et comparable d'un projet à l'autre.

---

## 2. Comment kodoneko estime le COSMIC (et pourquoi c'est une *estimation*)

### Le principe : analyse statique du code

kodoneko **lit le code sans l'exécuter** (analyse statique via AST pour Python,
tree-sitter pour Java/JS/TS) et reconnaît des motifs de framework :
un décorateur `@app.get` → un Entry ; un `session.add()` → un Write ; etc.

### Pourquoi ce n'est pas un comptage COSMIC *certifié*

C'est le point le plus important à comprendre. **La méthode COSMIC est définie
sur les besoins fonctionnels (FUR — *Functional User Requirements*), pas sur le
code.** Or certaines décisions de comptage demandent un **jugement humain** que
le code seul ne permet pas de trancher avec certitude :

- **Identifier les objets d'intérêt.** Le code manipule `Owner`, `ResponseEntity`,
  `Optional`… Lequel est l'objet métier ? kodoneko l'**infère** (heuristique),
  un humain le **sait**.
- **Distinguer un message d'une donnée.** Un endpoint avec trois `return`
  (erreur de validation, erreur de doublon, succès) ne produit **pas** trois
  sorties COSMIC : la règle officielle fusionne les messages d'erreur et de
  confirmation d'un même processus en **un seul Exit**. Le code, lui, montre
  trois `return`.

kodoneko ne peut donc pas garantir le chiffre exact. **Il produit une
estimation honnête, encadrée par un intervalle.**

### L'intervalle de confiance

Pour chaque fichier et chaque dépôt, kodoneko donne trois valeurs :

| Borne | Signification |
|-------|---------------|
| **basse** | hypothèse de **fusion maximale** des messages (le comptage COSMIC le plus strict possible) |
| **centrale** | meilleure estimation automatique (règle de fusion appliquée raisonnablement) |
| **haute** | comptage **brut** (chaque sortie détectée comptée séparément) |

La vraie valeur COSMIC se situe **dans cet intervalle**. Plus l'intervalle est
resserré, moins le code est ambigu. Les processus à comptage ambigu sont
**localisés explicitement** dans le rapport, pour que vous sachiez où le
jugement se joue.

> 📄 Le détail pédagogique de la règle de fusion est dans
> `PEDAGOGIE_FUSION_EXITS.md` (avec preuve par les exemples officiels COSMIC).

### Deux modes de comptage

| Mode | Comportement |
|------|--------------|
| `strict` | Aligné ISO/IEC 19761. Les appels HTTP **sortants** ne sont pas comptés (le service appelé n'est pas un utilisateur fonctionnel du système appelant). |
| `practical` *(défaut)* | Compte aussi les appels HTTP sortants comme 1 Write + 1 Read vers une frontière externe, pour refléter l'effort d'intégration réel. **Diverge volontairement** de la spec stricte. |

---

## 3. Choix d'implémentation assumés

Ces choix sont **déterministes et documentés** — c'est ce qui rend l'estimation
reproductible et auditable (essentiel pour un usage en suivi continu) :

- **Un mouvement = un acte syntaxique d'I/O.** Une boucle qui lit la base à
  chaque itération compte **1 Read**, pas N. On mesure le travail *conceptuel*,
  pas l'exécution runtime — conforme à l'esprit COSMIC.
- **Fusion des messages d'erreur/confirmation** par processus (règle 16-19a du
  manuel COSMIC), appliquée dans la borne basse et l'estimation centrale.
- **Exclusions par défaut** : tests, fichiers générés (`*.gen.ts`…), build,
  dépendances. La liste est explicite et inspectable (`exclusion_summary()`),
  et le rapport indique précisément ce qui a été compté ou écarté, et pourquoi.
- **Confiance par détecteur** : chaque mouvement détecté porte un niveau de
  confiance (élevée / moyenne) selon la fiabilité du motif reconnu.

---

## 4. Références officielles COSMIC

- **COSMIC Measurement Manual v5.0**, Part 2 — *Guidelines* (Sept. 2024).
  Règles de comptage, notamment les *Guidance Rules 16-19* sur la fusion des
  messages d'erreur/confirmation.
- **Case study C-REG v2.0.1** — exemple officiel de CRUD (gestion de
  professeurs/étudiants), utilisé pour valider le comptage des opérations
  create/read/update/delete.
- **Case study Web Advice Module v1.1.1** — exemple officiel illustrant le
  comptage des objets d'intérêt multiples et des messages.
- Norme **ISO/IEC 19761:2011** — définition normative de la méthode.

> kodoneko a été calibré et vérifié contre ces exemples officiels. Sur le
> contrôleur `OwnerController` de Spring PetClinic, l'estimation centrale tombe
> à ±1-2 CFP du comptage manuel validé contre C-REG.

---

## 5. Portée et limites

- **Langages supportés :** Python (`ast`, sans dépendance), JavaScript,
  TypeScript, TSX, Java (via tree-sitter). C#, Go, Kotlin, PHP, Ruby à venir.
- **Frontend pur** (canvas, jeux, éditeurs graphiques) : COSMIC **sous-compte**
  ces projets car la logique interne (géométrie, état UI) n'est pas un mouvement
  de données. Seules les frontières persistantes (`localStorage`, `fetch`,
  WebSocket) comptent. Pour ces projets, compléter avec d'autres mesures.
- **Estimation, pas certification.** Pour un livrable COSMIC officiel, l'analyse
  des FUR par un mesureur qualifié reste nécessaire. kodoneko fournit une
  estimation reproductible idéale pour le **suivi continu** et la comparaison.

---

## 6. Prérequis techniques

```bash
pip install -e kodoneko_metrics/ kodoneko_scanner/ kodoneko_temporal/
# Pour Java / JS / TS (Python fonctionne sans) :
pip install tree-sitter tree-sitter-java tree-sitter-javascript tree-sitter-typescript
```

La **section 8** (démonstration sur dépôts publics) nécessite en plus un accès
réseau pour `git clone`.


In [ ]:
# Bootstrap des chemins
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import setup_paths
setup_paths.setup(verbose=False)

from kodoneko_metrics.cosmic import CosmicAnalyzer
from kodoneko_scanner import scan_repo_cosmic
from kodoneko_temporal import (
    compute_cosmic_at_ref,
    compute_cosmic_delta_for_commit,
    compute_cosmic_delta_for_range,
)

## 1. Analyse d'un fichier — comprendre la détection

On commence par analyser un fichier en mémoire pour voir précisément ce que le détecteur identifie. Code FastAPI + SQLAlchemy + Redis + HTTP client :

In [ ]:
source = b'''
from fastapi import FastAPI
import redis, requests

api = FastAPI()
cache = redis.Redis()

@api.get("/users/{uid}")
async def get_user(uid: int):
    cached = cache.get(f"user:{uid}")
    if cached:
        return cached
    user = session.scalar(select(User).where(User.id == uid))
    cache.setex(f"user:{uid}", 60, user.to_json())
    return user.to_dict()

@api.post("/users")
async def create_user(payload: dict):
    user = User(**payload)
    session.add(user)
    session.commit()
    requests.post("https://api.email.com/welcome", json={"id": user.id})
    return {"id": user.id}
'''

analyzer = CosmicAnalyzer(mode='practical')
r = analyzer.analyze_source(source, 'python', 'backend.py')

print(f"Total CFP : {r.total_cfp}")
print(f"By type   : {r.by_type}")
print(f"By framework : {r.by_framework}")
print()
print("Détail des mouvements :")
for m in r.movements:
    print(f"  L{m.line:2d}  {m.type:5s}  [{m.detector:25s}]  {m.code_excerpt[:60]}")

### Strict vs Practical

Comparons les deux modes sur le même code. La différence vient des **calls HTTP sortants** (`requests.post`) qui comptent en mode practical mais pas en mode strict.

In [ ]:
for mode in ('strict', 'practical'):
    r = CosmicAnalyzer(mode=mode).analyze_source(source, 'python', 'backend.py')
    http_count = sum(1 for m in r.movements if 'http_client' in m.detector)
    print(f"Mode {mode:10s} : {r.total_cfp} CFP  (dont {http_count} HTTP client)")

## 2. Scan d'un repo complet

On scanne un repo réel. Pour la démo, on construit un mini-projet à 3 fichiers avec des frameworks différents.

In [ ]:
import tempfile, subprocess, os

DEMO = Path(tempfile.mkdtemp(prefix='kodoneko_cosmic_demo_'))

# Fichier 1 : Django views
(DEMO / 'views.py').write_text('''
from django.http import JsonResponse
from .models import Order

def list_orders(request):
    orders = Order.objects.filter(active=True).select_related("customer")
    return JsonResponse({"orders": [o.to_dict() for o in orders]})

def create_order(request):
    order = Order.objects.create(**request.POST.dict())
    return JsonResponse({"id": order.id}, status=201)
''')

# Fichier 2 : FastAPI + SQLAlchemy + Redis + HTTP
(DEMO / 'backend.py').write_text('''
from fastapi import FastAPI
import redis, requests

api = FastAPI()
cache = redis.Redis()

@api.get("/users/{uid}")
async def get_user(uid: int):
    cached = cache.get(f"user:{uid}")
    if cached: return cached
    user = session.scalar(select(User).where(User.id == uid))
    cache.setex(f"user:{uid}", 60, user.to_json())
    return user.to_dict()

@api.post("/users")
async def create_user(payload: dict):
    user = User(**payload)
    session.add(user); session.commit()
    requests.post("https://api.email.com/welcome", json={"id": user.id})
    return {"id": user.id}
''')

# Fichier 3 : Celery task + raw SQL + file I/O
(DEMO / 'tasks.py').write_text('''
from celery import Celery
app = Celery("tasks")

@app.task
def send_email(user_id):
    cursor.execute("SELECT email FROM users WHERE id = %s", (user_id,))
    row = cursor.fetchone()
    with open("/tmp/sent.log", "a") as f:
        f.write(f"sent to {row[0]}\\n")
''')

print(f"Repo de démo créé : {DEMO}")
print(f"Fichiers : {sorted(p.name for p in DEMO.iterdir())}")

In [ ]:
# Scan complet
report = scan_repo_cosmic(DEMO, mode='practical', use_git=False)

print(f"Scope      : {report.scope}")
print(f"Durée      : {report.duration_seconds*1000:.1f} ms")
print(f"Fichiers   : {len(report.files)} analysés, {len(report.skipped_files)} ignorés")
print(f"Total CFP  : {report.total_cfp}")
print()
print("Répartition par type de mouvement :")
for t, n in report.by_type.items():
    if n > 0:
        print(f"  {t:6s} : {n:3d} CFP")
print()
print("Répartition par framework détecté :")
for fw, n in sorted(report.by_framework.items(), key=lambda x: -x[1]):
    print(f"  {fw:15s} : {n:3d} CFP")

In [ ]:
# Top fichiers par CFP
print("Top fichiers par CFP :")
for fr in report.top_files(5):
    print(f"  {fr.total_cfp:3d}  {fr.path}")

## 3. Delta d'un commit

On construit un repo git avec 3 commits successifs pour mesurer l'apport CFP de chaque modification.

In [ ]:
# Construire un repo git de test
GIT_DEMO = Path(tempfile.mkdtemp(prefix='kodoneko_cosmic_git_'))

def run(args, cwd=GIT_DEMO):
    return subprocess.run(args, cwd=cwd, capture_output=True, text=True, check=True).stdout

run(['git', 'init', '-q', '-b', 'main'])
run(['git', 'config', 'user.email', 'demo@example.com'])
run(['git', 'config', 'user.name', 'Demo'])

# Commit 1 : 2 endpoints simples
(GIT_DEMO / 'app.py').write_text('''
from flask import Flask, jsonify
app = Flask(__name__)

@app.route("/health")
def health():
    return jsonify({"ok": True})

@app.route("/users/<int:uid>")
def get_user(uid):
    return jsonify(User.objects.get(id=uid).to_dict())
''')
run(['git', 'add', 'app.py'])
run(['git', 'commit', '-qm', 'Initial : health + get_user'])
run(['git', 'tag', 'v0.1.0'])

# Commit 2 : ajout endpoint POST
with (GIT_DEMO / 'app.py').open('a') as f:
    f.write('''
@app.route("/users", methods=["POST"])
def create_user():
    user = User.objects.create(**request.get_json())
    return jsonify({"id": user.id}), 201
''')
run(['git', 'add', 'app.py'])
run(['git', 'commit', '-qm', 'Add create_user endpoint'])

# Commit 3 : Redis cache + DELETE
(GIT_DEMO / 'app.py').write_text('''
from flask import Flask, jsonify, request
import redis
app = Flask(__name__)
cache = redis.Redis()

@app.route("/health")
def health():
    return jsonify({"ok": True})

@app.route("/users/<int:uid>")
def get_user(uid):
    cached = cache.get(f"u:{uid}")
    if cached: return jsonify(cached)
    user = User.objects.get(id=uid)
    cache.setex(f"u:{uid}", 60, user.to_json())
    return jsonify(user.to_dict())

@app.route("/users", methods=["POST"])
def create_user():
    user = User.objects.create(**request.get_json())
    return jsonify({"id": user.id}), 201

@app.route("/users/<int:uid>", methods=["DELETE"])
def delete_user(uid):
    user = User.objects.get(id=uid)
    user.delete()
    cache.delete(f"u:{uid}")
    return jsonify({"ok": True})
''')
run(['git', 'add', 'app.py'])
run(['git', 'commit', '-qm', 'Add Redis cache + DELETE'])
run(['git', 'tag', 'v0.2.0'])

print("Historique git :")
print(run(['git', 'log', '--oneline']))

In [ ]:
# Delta du dernier commit (HEAD)
head = run(['git', 'rev-parse', 'HEAD']).strip()
delta_head = compute_cosmic_delta_for_commit(GIT_DEMO, head)

print(f"Commit : {head[:8]}")
print(f"CFP avant : {delta_head.before.total_cfp}")
print(f"CFP après : {delta_head.after.total_cfp}")
print(f"Delta     : {delta_head.cfp_added:+d} CFP")
print()
print("Delta par type de mouvement :")
for t, d in delta_head.by_type_delta.items():
    if d != 0:
        sign = '+' if d > 0 else ''
        print(f"  {t:6s} : {sign}{d}")

**Interprétation** : le commit a ajouté un cache Redis (3 mouvements : get + setex + delete), un endpoint DELETE (1 entry + 1 exit + 1 read), et un get caché en plus dans get_user (1 read + 1 exit). Le delta total est de l'ordre de +8 à +10 CFP, ce qui colle à l'effort observable.

## 4. Delta entre deux versions (tags)

Mesure de l'apport CFP entre **v0.1.0** et **v0.2.0**.

In [ ]:
delta_v = compute_cosmic_delta_for_range(GIT_DEMO, since='v0.1.0', until='v0.2.0')

print(f"Scope     : {delta_v.scope}")
print(f"v0.1.0    : {delta_v.before.total_cfp} CFP  ({delta_v.before.by_type})")
print(f"v0.2.0    : {delta_v.after.total_cfp} CFP  ({delta_v.after.by_type})")
print(f"Delta     : {delta_v.cfp_added:+d} CFP")
print()
print("Décomposition du delta :")
for t, d in delta_v.by_type_delta.items():
    if d != 0:
        print(f"  {t:6s} : {d:+d}")
print()
print("Décomposition par langage :")
for lang, d in delta_v.by_language_delta.items():
    print(f"  {lang:10s} : {d:+d}")

## 5. Delta sur une plage de dates

Au lieu de tags, on peut spécifier des **dates**. C'est utile pour reporter l'activité d'un trimestre ou d'un mois, sans dépendre d'une discipline de tagging.

In [ ]:
from datetime import date, timedelta

# Pour la démo : du début du repo à aujourd'hui
today = date.today()
yesterday = today - timedelta(days=1)

# Note : sur ce repo de démo créé à l'instant, tous les commits sont "aujourd'hui"
# donc since=hier capture rien. Avec un vrai repo, ça fonctionne :
#   since=date(2025, 1, 1), until=date(2025, 6, 30)

delta_d = compute_cosmic_delta_for_range(GIT_DEMO, since=None, until=today)
print(f"Du début du repo à {today} :")
print(f"  Delta : {delta_d.cfp_added:+d} CFP")
print(f"  Par type : {delta_d.by_type_delta}")

## 6. TypeScript / JavaScript — exemples

Le détecteur JS/TS couvre les frameworks backend modernes (Express, NestJS, Fastify, Next.js Pages + App Router, Remix, SvelteKit), les ORMs (Prisma, TypeORM, Sequelize, Drizzle), Mongoose, Redis, fetch/axios, localStorage/IndexedDB, fs, WebSocket, BullMQ.

**Note** : nécessite `tree-sitter`, `tree-sitter-javascript`, `tree-sitter-typescript` (installés automatiquement avec `kodoneko-metrics`).

In [ ]:
# Test rapide : tree-sitter dispo ?
try:
    import tree_sitter_javascript  # noqa: F401
    import tree_sitter_typescript  # noqa: F401
    TS_AVAILABLE = True
except ImportError:
    TS_AVAILABLE = False
    print('⚠ tree-sitter-javascript / tree-sitter-typescript non installé.')
    print("  Installer : pip install tree-sitter tree-sitter-javascript tree-sitter-typescript")

In [ ]:
if not TS_AVAILABLE:
    print('Skip — tree-sitter manquant.')
else:
    # Exemple NestJS + Prisma + Redis + axios
    src_ts = b'''
import { Controller, Get, Post, Body, Param } from "@nestjs/common";
import axios from "axios";

@Controller("users")
export class UsersController {
    constructor(
        private prisma: PrismaService,
        private redis: Redis
    ) {}

    @Get(":id")
    async getUser(@Param("id") id: string) {
        const cached = await this.redis.get(`user:${id}`);
        if (cached) return JSON.parse(cached);
        const user = await this.prisma.user.findUnique({ where: { id } });
        await this.redis.setex(`user:${id}`, 60, JSON.stringify(user));
        return user;
    }

    @Post()
    async create(@Body() dto: any) {
        const user = await this.prisma.user.create({ data: dto });
        await axios.post("https://api.email.com/welcome", { id: user.id });
        return user;
    }
}
'''
    a = CosmicAnalyzer(mode="practical")
    r = a.analyze_source(src_ts, "typescript", "users.controller.ts")
    print(f"Total CFP    : {r.total_cfp}")
    print(f"By type      : {r.by_type}")
    print(f"By framework : {r.by_framework}")
    print()
    print("Détail :")
    for m in r.movements:
        print(f"  L{m.line:2d}  {m.type:5s}  [{m.detector:25s}]  {m.code_excerpt[:55]}")

In [ ]:
if not TS_AVAILABLE:
    print('Skip — tree-sitter manquant.')
else:
    # Exemple Next.js App Router avec plusieurs verbes
    src_next = b'''
import { NextResponse } from "next/server";

export async function GET(request) {
    const users = await prisma.user.findMany();
    return NextResponse.json(users);
}

export async function POST(request) {
    const body = await request.json();
    const user = await prisma.user.create({ data: body });
    return NextResponse.json(user, { status: 201 });
}

export async function DELETE(request) {
    const id = request.nextUrl.searchParams.get("id");
    await prisma.user.delete({ where: { id } });
    return NextResponse.json({ ok: true });
}
'''
    a = CosmicAnalyzer()
    r = a.analyze_source(src_next, "typescript", "app/api/users/route.ts")
    print(f"Total CFP : {r.total_cfp}")
    print(f"By type   : {r.by_type}")
    print(f"Routes Next.js App Router : {sum(1 for m in r.movements if 'nextjs' in m.detector)}")
    print()
    print("Détail :")
    for m in r.movements:
        print(f"  L{m.line:2d}  {m.type:5s}  [{m.detector:25s}]  {m.code_excerpt[:55]}")

In [ ]:
if not TS_AVAILABLE:
    print('Skip — tree-sitter manquant.')
else:
    # Exemple frontend : composant React avec localStorage + fetch
    src_react = b'''
import { useState, useEffect } from "react";

function UserPreferences() {
    const [theme, setTheme] = useState("light");

    useEffect(() => {
        const saved = localStorage.getItem("theme");
        if (saved) setTheme(saved);
    }, []);

    const save = async () => {
        localStorage.setItem("theme", theme);
        await fetch("/api/preferences", {
            method: "POST",
            body: JSON.stringify({ theme })
        });
    };

    return <button onClick={save}>Save</button>;
}
'''
    a = CosmicAnalyzer(mode="practical")
    r = a.analyze_source(src_react, "tsx", "UserPreferences.tsx")
    print(f"Total CFP : {r.total_cfp}")
    print(f"By framework : {r.by_framework}")
    print()
    print("Note : useState/useEffect (état React) NE SONT PAS comptés.")
    print("Seuls localStorage et fetch (frontières persistantes/réseau) le sont.")
    print()
    for m in r.movements:
        print(f"  L{m.line:2d}  {m.type:5s}  [{m.detector:25s}]  {m.code_excerpt[:55]}")

## 6bis. L'évolution COSMIC mois après mois

Les sections précédentes ont mesuré des *deltas* ponctuels (un commit, deux
tags, une plage de dates). Le moteur temporel générique permet d'aller plus
loin : reconstituer la **trajectoire complète** de la taille fonctionnelle, en
rejouant le comptage COSMIC à la fin de chaque mois.

C'est la même mécanique que pour les métriques de qualité — preuve que le moteur
temporel est agnostique à la métrique qu'on lui confie.

In [ ]:
from kodoneko_temporal import analyze_over_windows
from kodoneko_scanner import scan_repo_cosmic

# On réutilise un des dépôts publics clonés en section 8 si disponible,
# sinon le mini-dépôt git de démonstration créé plus haut.
demo_repo = None
try:
    demo_repo = repo_paths.get("spring-petclinic") or repo_paths.get("fastapi-template")
except NameError:
    pass

if demo_repo:
    cosmic = lambda path: scan_repo_cosmic(path, use_git=True).total_cfp
    serie = analyze_over_windows(demo_repo, analyzer=cosmic, strategy="monthly")
    print(f"Taille COSMIC (CFP) à la fin de chaque mois — {len(serie)} fenêtres :\n")
    maxv = max((p.result for p in serie), default=1) or 1
    for point in serie:
        barre = "█" * min(40, int(point.result / maxv * 40))
        print(f"  {point.label}  {point.result:>6} CFP  {barre}")
else:
    print("Aucun dépôt cloné disponible — exécutez d'abord la section 8.")

## 7. Configuration via TOML

La configuration COSMIC peut être placée dans `kodoneko.toml` :

```toml
[kodoneko.cosmic]
mode = "practical"                # ou "strict"
languages = ["python"]            # Restriction des langages
count_external_http = true        # Compter les calls HTTP sortants
```

(Note : l'intégration TOML est prévue pour la prochaine version.)

## 8. Limites et points d'attention

### Limites de la détection automatique

- **Faux positifs résiduels** : certains noms de méthodes ambiguës (ex: `find`, `get`, `delete`) peuvent générer des détections incorrectes selon le contexte. La confidence est ajustée en conséquence.
- **Frameworks non couverts** : les patterns sont implémentés pour les frameworks les plus courants (Django, Flask, FastAPI, SQLAlchemy, Redis, pymongo, Celery, Click, requests/httpx). Les frameworks plus rares peuvent ne pas être détectés.
- **Code dynamique** : les appels via `getattr`, `__getattr__`, ou indirections lourdes échappent à l'analyse statique. Conséquence : sous-comptage possible sur des architectures très dynamiques (rare).

### Limites du modèle COSMIC lui-même

- COSMIC mesure le **volume de mouvements**, pas la **complexité** de ce qui est fait avec eux. Un `cursor.execute("SELECT id FROM users")` et un `cursor.execute("WITH RECURSIVE ... complex CTE ...")` valent tous les deux 1 CFP.
- COSMIC ne mesure pas la **valeur technique** des changements (perf, sécurité, robustesse, etc.). Ces aspects sont mesurés séparément par le modèle VTA (Valeur Technique Ajoutée), non implémenté dans cette version.
- Un **refactoring pur** qui ne change ni les attributs ni les frontières du système donne un delta COSMIC = 0. C'est volontaire (voir docstring du module).

## 9. Cleanup

## 7bis. Détection TypeScript / JavaScript

Depuis la phase 1a, le module supporte aussi TypeScript et JavaScript. La détection couvre Express, NestJS, Fastify, Next.js (Pages + App Router), Remix, SvelteKit, Prisma, TypeORM, Sequelize, Drizzle, Mongoose, Redis (node-redis/ioredis), fetch/axios, fs/promises, localStorage/sessionStorage/IndexedDB, ws/socket.io, BullMQ.

**Prérequis** : `tree-sitter`, `tree-sitter-javascript` et `tree-sitter-typescript` doivent être installés. Ils sont inclus dans les dépendances obligatoires de `kodoneko-metrics`. Si la cellule suivante affiche un message d'absence, c'est que l'environnement Jupyter actuel n'a pas ces packages.


In [ ]:
# Vérification de la disponibilité de tree-sitter pour TS/JS
try:
    import tree_sitter, tree_sitter_javascript, tree_sitter_typescript
    TS_OK = True
    print('✓ tree-sitter JS/TS disponible')
except ImportError as e:
    TS_OK = False
    print(f'⚠ tree-sitter JS/TS indisponible : {e}')
    print('  Les cellules suivantes seront skip.')


In [ ]:
# Exemple : controller NestJS avec Prisma + Redis + axios
if TS_OK:
    ts_source = b'''
import { Controller, Get, Post, Body, Param } from '@nestjs/common';
import axios from 'axios';

@Controller('users')
export class UsersController {
    constructor(private prisma: PrismaService, private redis: Redis) {}

    @Get(':id')
    async findOne(@Param('id') id: string) {
        const cached = await this.redis.get(`user:${id}`);
        if (cached) return JSON.parse(cached);
        const user = await this.prisma.user.findUnique({ where: { id } });
        await this.redis.setex(`user:${id}`, 60, JSON.stringify(user));
        return user;
    }

    @Post()
    async create(@Body() dto: any) {
        const user = await this.prisma.user.create({ data: dto });
        await axios.post('https://api.email.com/welcome', { id: user.id });
        return user;
    }
}
'''
    r = analyzer.analyze_source(ts_source, 'typescript', 'users.controller.ts')
    print(f'Total CFP   : {r.total_cfp}')
    print(f'By type     : {r.by_type}')
    print(f'By framework: {r.by_framework}')
    print()
    print('Détail des mouvements :')
    for m in r.movements:
        print(f'  L{m.line:2d}  {m.type:5s}  [{m.detector:25s}]  {m.code_excerpt[:55]}')
else:
    print('(skip — tree-sitter JS/TS non installé)')


In [ ]:
# Exemple : route Next.js App Router avec deux verbes HTTP
if TS_OK:
    nextjs_source = b'''
import { NextResponse } from 'next/server';
import { db } from '@/lib/db';

export async function GET(request: Request, { params }: { params: { id: string } }) {
    const user = await db.select().from(users).where(eq(users.id, params.id));
    return NextResponse.json(user);
}

export async function POST(request: Request) {
    const body = await request.json();
    await db.insert(users).values(body);
    return NextResponse.json({ ok: true }, { status: 201 });
}
'''
    r = analyzer.analyze_source(nextjs_source, 'typescript', 'app/api/users/[id]/route.ts')
    print(f'CFP : {r.total_cfp} (attendu : 4 entries + reads + writes selon Drizzle)')
    print(f'By type     : {r.by_type}')
    print(f'By framework: {r.by_framework}')
    for m in r.movements:
        print(f'  L{m.line:2d}  {m.type:5s}  [{m.detector:30s}]  {m.code_excerpt[:55]}')
else:
    print('(skip — tree-sitter JS/TS non installé)')


## 7ter. Détection Java (Phase 1b)

Depuis la phase 1b, le module supporte aussi Java. La détection couvre :
- **Spring Web** : `@RestController` + `@GetMapping/@PostMapping/...` (tous les verbes HTTP)
- **Spring Data JPA** : `repository.findX/saveX/deleteX/...` (heuristique préfixe sur les méthodes dérivées)
- **JPA / Hibernate** : `entityManager.find/persist/merge/remove`, `createQuery`
- **JDBC** : `PreparedStatement.executeQuery/executeUpdate`, `Spring JdbcTemplate`
- **Spring Kafka / RabbitMQ / JMS** : `@KafkaListener`, `@RabbitListener` (Entry) + `KafkaTemplate.send`, `RabbitTemplate.convertAndSend` (Exit)
- **HTTP client** : `RestTemplate`, `WebClient`, `HttpClient` (Java 11+), `OkHttp`
- **Redis** : `redisTemplate.opsForValue/opsForHash/...` + `get/set/...`
- **File I/O** : `Files.readString/writeString`, `new FileReader/FileWriter`
- **CLI** : picocli `@Command`

**Prérequis** : `tree-sitter-java` doit être installé. Il est inclus dans les dépendances obligatoires depuis la v2.6.0.

In [ ]:
# Vérification de la disponibilité de tree-sitter pour Java
try:
    import tree_sitter, tree_sitter_java
    JAVA_TS_OK = True
    print('✓ tree-sitter Java disponible')
except ImportError as e:
    JAVA_TS_OK = False
    print(f'⚠ tree-sitter Java indisponible : {e}')
    print('  Les cellules suivantes seront skip.')


In [ ]:
# Exemple : Spring REST controller avec JPA + Redis + Kafka + RestTemplate
if JAVA_TS_OK:
    java_source = b'''
package com.example.users;

import org.springframework.web.bind.annotation.*;
import org.springframework.data.repository.CrudRepository;
import org.springframework.kafka.core.KafkaTemplate;
import org.springframework.data.redis.core.RedisTemplate;
import org.springframework.web.client.RestTemplate;

@RestController
@RequestMapping("/api/users")
public class UserController {
    private final UserRepository userRepository;
    private final RedisTemplate<String, User> redisTemplate;
    private final KafkaTemplate<String, UserEvent> kafkaTemplate;
    private final RestTemplate restTemplate;

    @GetMapping("/{id}")
    public User getOne(@PathVariable Long id) {
        User cached = redisTemplate.opsForValue().get("user:" + id);
        if (cached != null) return cached;
        User u = userRepository.findById(id).orElseThrow();
        redisTemplate.opsForValue().set("user:" + id, u);
        return u;
    }

    @PostMapping
    public User create(@RequestBody User u) {
        User saved = userRepository.save(u);
        kafkaTemplate.send("user-created", saved.getId().toString(), new UserEvent(saved));
        restTemplate.postForObject("https://api.email.com/welcome", saved, Void.class);
        return saved;
    }

    @DeleteMapping("/{id}")
    public void delete(@PathVariable Long id) {
        userRepository.deleteById(id);
        redisTemplate.delete("user:" + id);
    }
}
'''
    r = analyzer.analyze_source(java_source, 'java', 'UserController.java')
    print(f'Total CFP   : {r.total_cfp}')
    print(f'By type     : {r.by_type}')
    print(f'By framework: {r.by_framework}')
    print()
    print('Détail des mouvements :')
    for m in r.movements:
        print(f'  L{m.line:2d}  {m.type:5s}  [{m.detector:30s}]  {m.code_excerpt[:55]}')
else:
    print('(skip — tree-sitter Java non installé)')


In [ ]:
# Exemple : listener Kafka avec persistence JPA
if JAVA_TS_OK:
    listener_source = b'''
@Service
public class OrderListener {
    private final OrderRepository orderRepository;
    private final RestTemplate restTemplate;

    @KafkaListener(topics = "orders")
    public void onOrder(Order order) {
        Order saved = orderRepository.save(order);
        restTemplate.postForObject("https://shipping.api/dispatch", saved, Void.class);
    }

    @RabbitListener(queues = "refunds")
    public void onRefund(RefundEvent event) {
        Order o = orderRepository.findById(event.getOrderId()).orElseThrow();
        o.setRefunded(true);
        orderRepository.save(o);
    }
}
'''
    r = analyzer.analyze_source(listener_source, 'java', 'OrderListener.java')
    print(f'CFP : {r.total_cfp}')
    print(f'By type     : {r.by_type}')
    print(f'By framework: {r.by_framework}')
    for m in r.movements:
        print(f'  L{m.line:2d}  {m.type:5s}  [{m.detector:30s}]  {m.code_excerpt[:55]}')
else:
    print('(skip — tree-sitter Java non installé)')


## 8. Démonstration sur dépôts publics réels

Cette section clone plusieurs dépôts open-source de référence et les analyse de
bout en bout. Ils couvrent différents langages et frameworks de la matrice
supportée par kodoneko :

| Dépôt | Langage | Stack | Intérêt |
|-------|---------|-------|---------|
| **spring-petclinic** | Java | Spring Boot + Spring Data JPA | application de référence canonique de l'équipe Spring ; validée contre le comptage COSMIC manuel |
| **full-stack-fastapi-template** | Python + TS | FastAPI + SQLModel + React | template officiel de tiangolo (créateur de FastAPI) |
| **realworld (Django)** | Python | Django REST Framework | implémentation Django de l'app « Conduit » (clone de Medium) |

> 💡 **Le pattern RealWorld / Conduit** : la même application (« Conduit », un
> clone de Medium) est implémentée dans des dizaines de frameworks. Mesurer
> plusieurs implémentations du **même domaine fonctionnel** permet de comparer
> les estimations CFP entre langages — un bon test de cohérence.

> ⚠️ **Prérequis** : accès réseau (pour `git clone`) et **tree-sitter installé**
> pour Java/TS (`pip install tree-sitter tree-sitter-java tree-sitter-javascript
> tree-sitter-typescript`). Sans tree-sitter, seuls les fichiers Python sont
> analysés ; les fichiers Java/TS sont silencieusement ignorés.
>
> Vous pouvez activer/désactiver des dépôts dans la cellule ci-dessous en
> commentant les lignes correspondantes.


In [ ]:
# Clone des dépôts publics (idempotent : ne reclone pas si déjà présent)
import subprocess
from pathlib import Path

# Dépôts à analyser. Commentez une ligne pour désactiver un dépôt.
REPOS = {
    "spring-petclinic": "https://github.com/spring-projects/spring-petclinic.git",
    "fastapi-template": "https://github.com/fastapi/full-stack-fastapi-template.git",
    "realworld-django": "https://github.com/gothinkster/django-realworld-example-app.git",
}

CLONE_ROOT = Path(tempfile.gettempdir()) / "kodoneko_public_repos"
CLONE_ROOT.mkdir(exist_ok=True)

repo_paths = {}
for name, url in REPOS.items():
    dest = CLONE_ROOT / name
    if dest.exists():
        print(f"✓ {name} d\u00e9j\u00e0 pr\u00e9sent")
    else:
        print(f"⏳ Clone de {name} ...")
        r = subprocess.run(
            ["git", "clone", "--depth", "1", url, str(dest)],
            capture_output=True, text=True,
        )
        if r.returncode == 0:
            print(f"✓ {name} clon\u00e9")
        else:
            print(f"✗ \u00c9chec clone {name} : {r.stderr.strip()[:160]}")
            dest = None
    if dest and dest.exists():
        repo_paths[name] = dest

print(f"\n{len(repo_paths)} d\u00e9p\u00f4t(s) pr\u00eat(s) \u00e0 analyser.")


In [ ]:
# Scan COSMIC des deux repos (exclusions par défaut : tests, générés, build)
scan_reports = {}
for name, path in repo_paths.items():
    report = scan_repo_cosmic(path, mode="practical")  # use_git=True par défaut
    scan_reports[name] = report
    print(f"=== {name} ===")
    print(f"  Total CFP  : {report.total_cfp}")
    print(f"  Fichiers   : {len(report.files)} analysés, {len(report.skipped_files)} exclus")
    print(f"  Par langage : {dict(report.by_language)}")
    print(f"  Par type    : {dict(report.by_type)}")
    print()


### Traçabilité des exclusions

Les exclusions sont **déterministes et auditables** : on peut lister exactement
quels fichiers ont été comptés et lesquels ont été écartés (et pourquoi). C'est
essentiel pour justifier un comptage devant un tiers.


In [ ]:
# Audit des exclusions sur un repo
if "spring-petclinic" in scan_reports:
    rep = scan_reports["spring-petclinic"]
    print("Résumé des exclusions :", rep.exclusion_summary())
    print(f"\nFichiers analysés : {len(rep.analyzed_files)}")
    for f in rep.analyzed_files[:8]:
        print(f"  ✓ {Path(f).name}")
    print(f"\nExemples d'exclusions :")
    for path, reason in rep.excluded_files[:8]:
        print(f"  ✗ {Path(path).name:40s} [{reason}]")


### Analyse enrichie avec intervalle de confiance

C'est la vue façon manuel COSMIC : par functional process, le détail des data
movements, et un **intervalle** encadrant la valeur (le comptage statique ne peut
pas trancher tous les jugements COSMIC, notamment la fusion des messages
erreur/confirmation — voir `PEDAGOGIE_FUSION_EXITS.md`).


In [ ]:
# Analyse enrichie : intervalle de confiance au niveau repo
from kodoneko_metrics.cosmic import analyze_repo_enriched

for name, report in scan_reports.items():
    enriched = analyze_repo_enriched(report)
    iv = enriched.interval
    print(f"=== {name} ===")
    print(f"  Taille estimée : {iv.label()}")
    print(f"    borne basse {iv.low}  |  central {iv.central}  |  borne haute {iv.high}")
    print(f"  FP avec incertitude de fusion (L10) : {len(enriched.uncertain_fps)}")
    print()


In [ ]:
# Focus : détail façon manuel COSMIC sur un fichier (OwnerController)
from kodoneko_metrics.cosmic import analyze_enriched, render_text

if "spring-petclinic" in repo_paths:
    candidates = list(repo_paths["spring-petclinic"].rglob("OwnerController.java"))
    if candidates:
        fr = CosmicAnalyzer(mode="practical").analyze_file(str(candidates[0]))
        analysis = analyze_enriched(fr)
        print(render_text(analysis))
    else:
        print("OwnerController.java introuvable")


### Rapport HTML pédagogique

Le rendu HTML est conçu pour être **lu par un humain** : badges colorés par type
de mouvement, encadré expliquant l'intervalle, avertissement sur le statut
d'estimation, et bandeau signalant les processus à comptage ambigu.

C'est le format à privilégier pour **partager une analyse** ou l'intégrer dans
un rapport.

In [ ]:
# Rapport HTML pédagogique d'un fichier (affiché dans le notebook)
from kodoneko_metrics.cosmic import render_html
from IPython.display import HTML, display

if "spring-petclinic" in repo_paths:
    candidates = list(repo_paths["spring-petclinic"].rglob("OwnerController.java"))
    if candidates:
        fr = CosmicAnalyzer(mode="practical").analyze_file(str(candidates[0]))
        analysis = analyze_enriched(fr)
        display(HTML(render_html(analysis)))


### Synthèse HTML d'un dépôt entier

Vue d'ensemble : estimation globale, nombre de processus ambigus, et classement
des fichiers par taille estimée.

In [ ]:
# Synthèse HTML au niveau dépôt
from kodoneko_metrics.cosmic import render_repo_html

if "spring-petclinic" in scan_reports:
    enriched = analyze_repo_enriched(scan_reports["spring-petclinic"])
    display(HTML(render_repo_html(enriched)))


### Export d'un rapport HTML autonome

Avec `standalone=True`, le rapport est un document HTML complet, partageable
tel quel (ouvrable dans un navigateur, joignable à un e-mail).

In [ ]:
# Export d'un rapport HTML autonome sur disque
if "spring-petclinic" in scan_reports:
    enriched = analyze_repo_enriched(scan_reports["spring-petclinic"])
    out = Path("rapport_cosmic_petclinic.html")
    out.write_text(render_repo_html(enriched, standalone=True), encoding="utf-8")
    print(f"Rapport exporté : {out.resolve()}")
    print("Ouvrable directement dans un navigateur.")


In [ ]:
import shutil
for d in (DEMO, GIT_DEMO):
    if d.exists():
        shutil.rmtree(d)
        print(f'Supprimé : {d}')

---

**Récapitulatif des points clés :**

1. **COSMIC = compter les mouvements de données** : entry / exit / read / write, sans pondération de complexité.
2. **Mode strict vs practical** : le mode practical inclut les calls HTTP sortants (+1 Write +1 Read), au prix d'une divergence avec la spec ISO. À documenter dans tout rapport.
3. **Trois granularités** : scan repo (`scan_repo_cosmic`), delta commit (`compute_cosmic_delta_for_commit`), delta version/dates (`compute_cosmic_delta_for_range`).
4. **Mécanique de détection** : AST stdlib pour Python (rapide, sans dépendance externe), tree-sitter pour les autres langages (prévu pour JS/TS et Java).
5. **Limites assumées** : faux positifs résiduels sur méthodes ambiguës (confidence ajustée), refactoring pur = delta 0 (volontaire), COSMIC ne mesure pas la complexité ni la valeur technique.